# Synergy quickstart: arbitrary groups and nodes

This notebook demonstrates a small synthetic EEG-style example. Replace the synthetic `prepared_signals` dictionary with your preprocessed arrays. Each array is **nodes × samples**, and every subject must use the same node order. The analysis works with any group names, node labels, and number of groups. No patient cohort or atlas is assumed.

Install with `python -m pip install "./synergy[analysis]"`. Add `fast` for accelerated pairwise CCS calculations.

## Preprocessing comes first

Phi_ID measures temporal dependence in the signal you provide. The toolbox does not clean raw data, infer a sampling rate, or decide which artifacts to remove. Record the preprocessing pipeline, `tau`, and the physical lag (`tau × TR` for fMRI or `tau / sampling_rate` for EEG). Apply the same steps and node order to every subject being compared.

| fMRI | EEG |
|---|---|
| Extract time series from a stated parcellation; document atlas and ROI order. | Choose sensor or source-space signals; document montage, reference, and channel order. |
| Address motion, nuisance regressors, temporal filtering, censoring, and short/constant series. | Address bad channels, line noise, eye/muscle artifacts, filtering, re-referencing, and epoch selection. |
| State whether signals are BOLD or HRF-deconvolved. HRF smoothing and regional HRF differences can affect lagged measures. | State sampling rate and any downsampling. Volume conduction and source leakage can create spurious shared structure. |
| Choose `tau` in TR samples and keep acquisition/preprocessing choices comparable across groups. | Choose `tau` in samples after preprocessing; use comparable epochs, lengths, and conditions across groups. |

For both modalities, avoid nonfinite or near-constant channels. Check that the chosen window is reasonably stationary and contains enough samples for covariance estimation. Do not compare groups processed with different pipelines without treating that as an explicit analysis factor.

In [ ]:
from pathlib import Path
import numpy as np
from synergy import pairwise_atoms
from synergy.analysis.workflow import SubjectRecord, analyze_subjects
from synergy.statistics.inference import infer_groups
from synergy.plotting.general import render_figures

rng = np.random.default_rng(7)
node_labels = ["Fz", "Cz", "Pz", "Oz", "Tz"]
node_systems = ["frontal", "central", "parietal", "occipital", "temporal"]
sampling_rate_hz = 250  # Replace with your acquisition rate
lag_ms = 12
tau = round(lag_ms * sampling_rate_hz / 1000)
print("tau =", tau, "samples")

## Load your prepared signals

The synthetic dictionary below only makes the notebook runnable without private data. Replace its arrays with your already preprocessed fMRI or EEG time series. One common pattern is `np.load(path)` for each subject and a metadata CSV containing `subject_id`, `group`, and `path`. Validate the shape and node order at loading time.

In [ ]:
prepared_signals = {}
for group in ("control", "meditation"):
    for i in range(3):
        signal = rng.normal(size=(len(node_labels), 500))
        if group == "meditation":
            signal[1] += 0.25 * signal[0]
        prepared_signals[f"{group}_{i+1}"] = (group, signal)

assert all(x.shape[0] == len(node_labels) for _, x in prepared_signals.values())
assert all(np.isfinite(x).all() for _, x in prepared_signals.values())

## Compute subject matrices

`pairwise_atoms` returns `rtr` and `sts` matrices for each requested estimator. Use `methods=("mmi", "ccs")` for both estimators; the MMI-only example below runs quickly. For an existing Phi_ID dataset, load its matrix files and construct `SubjectRecord` directly. Optional comparison networks (for example FC, SC, or EEG coherence) can be passed as `connectivity={"name": square_matrix}`.

In [ ]:
records = []
for subject_id, (group, signal) in prepared_signals.items():
    atoms = pairwise_atoms(signal, tau=tau, methods=("mmi",))
    records.append(SubjectRecord(subject_id=subject_id, group=group, atoms=atoms))
print(len(records), "subjects")

## Summarize and test groups

Choose any reference group; it need not be named `control`. `node_systems` is optional. When supplied, the analysis also computes system-level summaries. The paired atom tests use sign-flip permutations; group contrasts use Welch tests and Benjamini–Hochberg FDR correction. Small synthetic examples have little inferential power, so interpret their p values only as a demonstration.

In [ ]:
analysis = analyze_subjects(
    records,
    node_labels=node_labels,
    node_systems=node_systems,
    group_order=["control", "meditation"],
    reference_group="control",
)
inference = infer_groups(analysis, n_perm=999, seed=42)
display(analysis.subject.head())
display(inference.group_nodal.head())

## Create figures

The general renderer adapts its panels to the supplied groups and nodes. It generates global atom, paired atom, graph metric, nodal gradient, system gradient, and regional effect figures when those inputs are available. It generates similarity panels only when `SubjectRecord.connectivity` is supplied. The saved files go to the directory you choose; running this cell is optional.

In [ ]:
output_dir = Path("synergy_figures")
made = render_figures(analysis, output_dir, inference=inference, formats=("png", "pdf"))
print("Created", len(made), "figure families")
print("\n".join(made))

## Existing DoC figure preset

The package also retains the original AAL90 figure workflow as a dataset-specific preset. It expects the repaired matrix layout, AAL atlas, and fsaverage assets described in the README. Use it only when that data contract applies; the general API above does not require those assets.